# Module 3 Assignment: Window Functions, CTEs, and a Second Way In

**Name:**  
**Team:**  
**Team dataset:** `bigquery-public-data.[dataset]`

Get this file with `git pull` in your team repo (it is at `templates/M3-starter.ipynb`), then copy it to `members/<your-netid>/M3.ipynb` and work there. Your netid is your Canvas login, not your GitHub handle.

Three parts, in order, plus the same task done a second way from the terminal. **Part A is your pre-AI attempt: commit and push when Part A is done, before you start Part B** (commit message `M3 Part A: pre-AI attempt`). That commit is a timestamp on your own reasoning and is what the pre-AI rubric row is graded from. Do not edit Part A afterwards; corrections go in Part B.

**Submitting (new from this module):** upload this `.ipynb` on the Module 3 Assignment page with **File Upload**, then paste the **commit permalink** to the file (press `y` on the file page in GitHub; the URL carries a 40-character commit id) into the **Comments** box under the upload, and submit once. The URL box is not used. Steps: *Submitting with a GitHub Repo*.

**Cost guard:** name your columns; never `SELECT *` on a large table. Read the scan estimate in the console before you run.


## Worked example: the running total that did not run

Both queries below were run on 2026-09-08 against Iowa Liquor Sales for calendar 2024. Run them yourself before Part A; the point is the check, not the numbers.

```sql
WITH monthly AS (
  SELECT DATE_TRUNC(date, MONTH) AS month, SUM(sale_dollars) AS revenue
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
  GROUP BY month
)
SELECT FORMAT_DATE('%Y-%m', month)              AS m,
       ROUND(revenue)                            AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY month)) AS running_total,   -- remove ORDER BY and every row shows 447,235,414
       ROUND(SUM(revenue) OVER ())               AS grand_total
FROM monthly
ORDER BY month
```

Expected: `2024-01` revenue 32,972,813 equals its own running total; `2024-12` running total 447,235,414 equals `grand_total`. **The ten-second check:** row 2 differs from row 1, and the last running total equals the grand total. A window that "runs clean" with the grand total on every row has not run at all; it is missing the `ORDER BY` inside `OVER`.

```sql
WITH store_rev AS (
  SELECT county, store_number, ANY_VALUE(store_name) AS store_name, SUM(sale_dollars) AS revenue
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31' AND county IN ('POLK', 'LINN')
  GROUP BY county, store_number
)
SELECT county, store_name, ROUND(revenue) AS revenue,
       ROW_NUMBER() OVER (PARTITION BY county ORDER BY revenue DESC) AS rank_in_county
FROM store_rev
QUALIFY rank_in_county <= 3
ORDER BY county, rank_in_county
```

Expected: Linn 1 Benz Distributing 5,157,381; Polk 1 Hy-Vee #3 / BDI / Des Moines 14,409,325; the rank goes 1, 2, 3 and then **restarts at 1** when the county changes. If it does not restart, the `PARTITION BY` is wrong.

The `WITH` clause is a CTE: a named step you can read on its own. Part A2 asks you to turn a query with a nested subquery into this shape.


## Part A. Pre-AI attempt (no AI yet)

**Stakeholder question:** [one sentence: who is asking, and what do they want to know?]

**Table(s) involved:** [table, and the grain of one row]

Write your own first try, by hand. Wrong output and errors are fine and expected; Part A is graded on effort and reasoning, not correctness. **Above the window query, write one line predicting what its first three rows should show, before you run it.** You will check the prediction in the same cell after it runs.

When you finish Part A: **Kernel, Restart & Run All, save, `git add`, `git commit -m "M3 Part A: pre-AI attempt"`, `git push`.** Then continue to Part B without changing these cells.


In [ ]:
# Setup: run once. Uses the personal Google account from the BigQuery Sandbox page.
# See Jupyter Environment Setup, "Connect Your Notebook to BigQuery".
import pandas_gbq
PROJECT_ID = "your-sandbox-project-id"   # from the BigQuery console, top-left project selector

def run(sql):
    """Run a SQL string against BigQuery and return a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=PROJECT_ID)


In [ ]:
# A1. Window function: a running total, a rank that restarts per group, or a period-over-period comparison (LAG).
# Question: [what question does this answer, for your stakeholder?]
# Prediction (write BEFORE running): [the first three rows should show ... because ...]
# Window check (fill AFTER running): row 2 differs from row 1? [yes/no]. Last running total equals the grand total, or the rank restarts at 1 per group? [yes/no]
run("""
WITH base AS (
  SELECT [group column], [period column], SUM([measure]) AS [measure_total]
  FROM `bigquery-public-data.[dataset].[table]`
  WHERE [condition]
  GROUP BY [group column], [period column]
)
SELECT [group column], [period column], [measure_total],
       SUM([measure_total]) OVER (PARTITION BY [group column] ORDER BY [period column]) AS running_total
FROM base
ORDER BY [group column], [period column]
""")


In [ ]:
# A2. CTE refactor. First, a multi-step query the old way: one query nested inside another.
# Question: [ ... ]
run("""
SELECT [columns]
FROM (
  SELECT [columns], [aggregate]
  FROM `bigquery-public-data.[dataset].[table]`
  WHERE [condition]
  GROUP BY [columns]
) AS inner_step
WHERE [filter on the aggregate]
ORDER BY [column]
LIMIT 20
""")


In [ ]:
# A2, refactored. The same question with a named step (WITH ...). Same rows must come back.
# Row check: nested version = [ n ] rows, CTE version = [ n ] rows. Same? [yes/no]
run("""
WITH step_one AS (
  SELECT [columns], [aggregate]
  FROM `bigquery-public-data.[dataset].[table]`
  WHERE [condition]
  GROUP BY [columns]
)
SELECT [columns]
FROM step_one
WHERE [filter on the aggregate]
ORDER BY [column]
LIMIT 20
""")


## Part B. Queries with AI

Now bring AI in: ask for what you actually want, read what it gives you, keep / change / reject, and run it. One cell per query. Every window query carries a filled `# Window check:` line; every CTE refactor carries a `# Row check:` line comparing it with the version it replaced. If AI split one step into two CTEs, say whether the result changed. If it added a column you did not ask for, decide whether it belongs. An AI will not run these checks for you, and neither will an error message.


In [ ]:
# B1. Window function, with AI.
# Question: [ ... ]
# What I asked AI, in one line: [ ... ]
# Window check: row 2 differs from row 1? [yes/no]. Last row equals the grand total, or rank restarts per group? [yes/no]. Compared with A1: [same result / different, because ...]
run("""

""")


In [ ]:
# B2. CTE refactor, with AI.
# Question: [ ... ]
# What I asked AI, in one line: [ ... ]
# Row check: nested version = [ n ] rows, CTE version = [ n ] rows. Same? [yes/no]. Compared with A2: [ ... ]
run("""

""")


In [ ]:
# B3 (optional). One more window or CTE query your stakeholder would ask for.
# Question: [ ... ]
# Window check or Row check: [ ... ]
run("""

""")


## Part B, second way in: the same query from the `bq` command line

Run your B1 query (or A1 if you kept your own) from the VS Code terminal with the `bq` CLI, and paste the command and the first rows of output into the cell below. Setup and the first command are on the *bq Command Line Setup* page. The rows must match the notebook run; if they do not, say which run you trust and why.

```
bq query --nouse_legacy_sql --max_rows 5 '
[your B1 SQL here]
'
```


**Command I ran:**

```
[paste the exact bq command]
```

**First rows of output:**

```
[paste the output, or attach a screenshot named members/<netid>/M3-bq.png and reference it here]
```

**Match with the notebook run?** [yes / no, and why]


## Part C. AI Attribution Log

One row per AI-mediated step that changed what you shipped (decisions, not keystrokes). Name the tool, quote or closely paraphrase what you asked, and say what you kept, changed, or rejected. "It looked right" is not a verification; the window check or a row count is.

| # | Where (which query) | Tool (and model if known) | What I asked for | What it gave me | What I did with it (accepted / edited / rejected) | How I verified it |
|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |


## Reflection: which way in felt natural, and why

[Which access pattern, notebook or `bq` command line, did you use for which task this week, and why? Three to five sentences, tied to what actually happened: a query that was easier to iterate in one place, output that was easier to read in the other, a setup step that cost time.]


## Before submitting

- A1 and B1 each have a filled `# Window check:` line; A2 and B2 each have a filled `# Row check:` line.
- The `bq` section has the command, the output, and the match line.
- Part C names the tool and what you asked; the Reflection answers which way in and why.
- Kernel, Restart & Run All; every cell shows the output you intend the grader to see.
- Save, `git add`, `git commit -m "M3 complete"`, `git push`.
- On GitHub, open `members/<your-netid>/M3.ipynb`, press `y`, copy the URL with the 40-character commit id.
- Canvas, Module 3 Assignment: **File Upload** this `.ipynb`, paste the permalink in the **Comments** box, submit once.
